# Basic-Lite TRACE — Fase 5: Razonamiento Temprano 2B (`<think>` + TIR + GRPO corto)

**Copyright © 2026 Miguel19x. Todos los derechos reservados (`LICENSE.txt`).**

Este notebook está diseñado para ejecutarse en **Kaggle (GPU T4 16 GB o 2× T4)** y tarda aproximadamente **30–55 minutos** en total:
1. Clona `Basic-Lite-TRACE` e instala `transformers>=4.51`, `peft` y `accelerate`.
2. Genera un dataset verificado en sandbox (~700 trazas `<think>` + `<tool_call>` + `<answer>` con ≥15% de mezcla general y **0 fuga** con el banco oculto 4B).
3. Ajusta `microsoft/bitnet-b1.58-2B-4T-bf16` (pesos maestros oficiales BF16 para fine-tuning) con **LoRA (`r=16`) + SFT enmascarado + GRPO/RLVR corto** (usando `disable_adapter()` como referencia KL sin duplicar VRAM) y hace *merge* de los adaptadores.
4. Convierte el modelo resultante directamente a **`Paged BitCOS v1.1`** (`tools/import_bitnet.py`, ~1.2 GB) listo para descargar y evaluar en CPU local (`i3-1115G4`, `OMP_NUM_THREADS=2`) con `scripts/run_bench_4b.py`.

In [ ]:
!nvidia-smi
!pip -q install --upgrade "transformers>=4.51.0" peft accelerate safetensors

import os

REPO_DIR = "/kaggle/working/Basic-Lite-TRACE"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/Miguel19x/Basic-Lite-TRACE.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull --ff-only
%cd {REPO_DIR}

In [ ]:
# 1. Verificar tests unitarios de Fase 5 antes de gastar GPU
!OMP_NUM_THREADS=2 PYTHONPATH=. pytest tests/test_phase5_reasoning.py tests/test_bench_4domains.py -v

In [ ]:
# 2. Entrenamiento QAT-STE (W1.58A8) SFT (<think> + TIR) + GRPO corto sobre BitNet-b1.58-2B-4T-bf16
# --qat-ste garantiza 0 choque de cuantizacion al exportar a Paged BitCOS v1.1 (~25-35 min en 1x T4)
!PYTHONPATH=. python3 scripts/train_phase5_sft_grpo.py \
    --backend hf-2b \
    --model-id microsoft/bitnet-b1.58-2B-4T-bf16 \
    --use-lora --qat-ste --lora-r 16 --lora-alpha 32 \
    --num-math 360 --num-code 180 --num-factual 120 --general-ratio 0.15 \
    --sft-steps 560 --grad-accum 8 --lr 5e-5 --max-seq-len 384 \
    --grpo-steps 24 --grpo-group-size 4 --grpo-max-new-tokens 96 \
    --export-dataset /kaggle/working/fase5_tir_dataset.jsonl \
    --out-dir /kaggle/working/bitnet_2b_phase5_hf

In [ ]:
# 3. Exportar el modelo fusionado a Paged BitCOS v1.1 (W1.58 + embeddings FP16) + TRTK v2
!PYTHONPATH=. python3 tools/import_bitnet.py \
    --model-dir /kaggle/working/bitnet_2b_phase5_hf \
    --output-model /kaggle/working/bitnet_2b_phase5_fp16.bin \
    --output-tokenizer /kaggle/working/bitnet_2b_phase5_tok_v2.bin \
    --embedding-dtype float16
!ls -lh /kaggle/working/bitnet_2b_phase5_fp16.bin /kaggle/working/bitnet_2b_phase5_tok_v2.bin

## Evaluación local en `i3-1115G4` (Puerta de Salida de Fase 5)

Descarga `bitnet_2b_phase5_fp16.bin` y `bitnet_2b_phase5_tok_v2.bin` a `checkpoints/` en tu equipo local y ejecuta:
```bash
OMP_NUM_THREADS=2 PYTHONPATH=. venv/bin/python scripts/run_bench_4b.py \
    --model checkpoints/bitnet_2b_phase5_fp16.bin \
    --tokenizer checkpoints/bitnet_2b_phase5_tok_v2.bin \
    --max-new-tokens 128 \
    --json-out results/fase5_bench_2b_c1_c3.json
```
Compara contra el baseline de Fase 4B (`results/fase4b_bench_2b_c1_c3.json`: `C1=25%`, `C2=50%`, `C3=37.5%`).